## 1. Configuração e Ligação ao Delta Lake (MinIO)

In [ ]:
import os
import boto3
from botocore.config import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

# Carregar variáveis de ambiente (as mesmas que já usas no csv_to_delta)
load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'miniopass')
BRONZE_BUCKET    = 'datalake'
BRONZE_PATH      = f's3a://{BRONZE_BUCKET}/bronze'

s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name='us-east-1',
    config=Config(s3={'addressing_style': 'path'}),
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f"Bucket '{BRONZE_BUCKET}' já existe.")
except Exception:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f"Bucket '{BRONZE_BUCKET}' criado com sucesso.")

spark = (
    SparkSession.builder
    .appName('DML_Operacoes_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-core_2.12:2.4.0')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)

# Caminho para a tabela students no bucket bronze da camada datalake
caminho_students = f'{BRONZE_PATH}/students'

print('Sessão Spark Iniciada. Prontos para DML no Delta Lake!')

ModuleNotFoundError: No module named 'dotenv'

## 2. Operação de INSERT

In [ ]:
print('--- 1. Executar INSERT ---')
# Criar um DataFrame com um novo aluno (ID 4), mantendo a mesma estrutura do teu banco
novo_aluno = spark.createDataFrame([
    (4, 'Aluno DML Teste', 'S2004', 'M', '2010-01-01 00:00:00', 1)
], ['id', 'name', 'goverment_code', 'gender', 'birth_date', 'class_id'])

# O INSERT no Delta é feito através de "append"
novo_aluno.write.format('delta').mode('append').save(caminho_students)

# Verificar se inseriu corretamente
spark.read.format('delta').load(caminho_students).filter(col('id') == 4).show()

## 3. Operação de UPDATE

In [ ]:
print("--- 2. Executar UPDATE ---")
# Carregar a tabela Delta para habilitar DML
dt_students = DeltaTable.forPath(spark, caminho_students)

# Atualizar a class_id de 1 para 3 para o Aluno DML Teste
dt_students.update(
    condition = col("id") == 4,
    set = { "class_id": lit(3) } 
)

# Verificar se atualizou
spark.read.format("delta").load(caminho_students).filter(col("id") == 4).show()

## 4. Operação de DELETE

In [ ]:
print("--- 3. Executar DELETE ---")
# Apagar o aluno com ID 4
dt_students.delete(condition = col("id") == 4)

# Verificar se foi apagado (deve retornar vazio)
spark.read.format("delta").load(caminho_students).filter(col("id") == 4).show()

## 5. Auditoria e Histórico de Operações (Delta History)

In [ ]:
print("--- 4. HISTÓRICO DE OPERAÇÕES (PROVA DO DML) ---")
# O Delta regista todas as operações. Isto prova ao professor que fizeste tudo.
dt_students.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)